In [15]:
import pandas as pd
pd.set_option('display.max_columns', None)
import altair as alt
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib import colormaps
import matplotlib as mpl
import numpy as np

import fastf1 as ff1
import fastf1.plotting

import ipywidgets as widgets
from IPython.display import display, clear_output


import logging
logging.getLogger('fastf1').setLevel(logging.ERROR)
import warnings
warnings.filterwarnings('ignore')


## Data Exploration Hierarchy (Granular)

**By Year**
- Retrieve list of events for a given year

**By Race**
- For a selected event, retrieve list of sessions

**By Driver**
- Load session data for a selected driver

**By Lap**
- Access Lap data for a selected driver

> When user adjusts upper level of hierarchy, cache should be handled correctly

**For Cache Management**
```
ff1.Cache.get_cache_info()
ff1.Cache.clear_cache()
```

In [11]:
def get_schedule(year: int):
    """
    Returns the list of events for a given year (2018-2024)
    """
    return pd.DataFrame(ff1.get_event_schedule(year))

# TODO: Create dropdown of event_names from get_schedule(year)
def load_race_session(year: int, event_name: str):
    """
    Returns the details of the given race event (Races only)
    """
    session = ff1.get_session(year, event_name, 'Race')
    session.load()
    return session

# TODO: Allow user to select driver name, which then maps to driver code
def get_driver_laps(session, driver_code: str):
    """
    Returns the laps for a given driver in a given session
    """
    laps = session.laps.pick_driver(driver_code)
    return laps

def get_lap_telemetry(lap):
    """
    Returns the telemetry for a given lap
    """
    tel = lap.get_car_data().add_distance()
    return tel


## Interaction Logic

- User selects a year
- User selects an event
- User selects a driver
- User selects a lap
- User selects a telemetry variable

In [34]:
# 1. Year Dropdown
year_dropdown = widgets.Dropdown(
    options=sorted(range(2018, 2025), reverse=True),
    description='Year:'
)

race_dropdown = widgets.Dropdown(description='Race:')
driver_dropdown = widgets.Dropdown(description='Driver:')
lap_dropdown = widgets.Dropdown(description='Lap:')

output = widgets.Output()

def update_races(change):
    year = year_dropdown.value
    schedule = get_schedule(year)
    race_dropdown.options = schedule['EventName'].tolist()
    race_dropdown.value = race_dropdown.options[0]


def update_drivers(change):
    year = year_dropdown.value
    race = race_dropdown.value
    session = load_race_session(year, race)
    results = session.results
    driver_dropdown.options = results['Abbreviation'].tolist()
    driver_dropdown.value = driver_dropdown.options[0]

def update_laps(change):
    year = year_dropdown.value
    race = race_dropdown.value
    driver = driver_dropdown.value
    session = load_race_session(year, race)
    driver_laps = get_driver_laps(session, driver)
    lap_dropdown.options = driver_laps['LapNumber'].tolist()
    lap_dropdown.value = lap_dropdown.options[0]


def show_telemetry(change):
    output.clear_output()
    year = year_dropdown.value
    race = race_dropdown.value
    driver = driver_dropdown.value
    lap_num = lap_dropdown.value

    session = load_race_session(year, race)
    driver_laps = get_driver_laps(session, driver)
    lap = driver_laps.loc[driver_laps['LapNumber'] == lap_num].iloc[0]
    
    # Get telemetry including position data
    tel = lap.get_telemetry().drop(columns=['Time', 'SessionTime'])

    def plot_colored_track(col, title, cmap='viridis'):
        x = tel['X'].to_numpy()
        y = tel['Y'].to_numpy()
        color = tel[col].to_numpy()

        points = np.array([x, y]).T.reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)

        lc = LineCollection(segments, cmap=cmap, norm=plt.Normalize(np.nanmin(color), np.nanmax(color)))
        lc.set_array(color)
        lc.set_linewidth(4)

        fig, ax = plt.subplots(figsize=(10, 6))
        ax.add_collection(lc)
        ax.plot(x, y, color='lightgray', linewidth=1, alpha=0.5)
        ax.axis('equal')
        ax.set_title(f'{driver} - Lap {lap_num} ({col})')
        ax.axis('off')
        plt.show()

    with output:
        print(f"{driver} - Lap {lap_num} ({race} {year})")
        plot_colored_track('Speed', 'Speed (km/h)', cmap='plasma')
        plot_colored_track('Throttle', 'Throttle (%)', cmap='Greens')
        plot_colored_track('nGear', 'Gear', cmap='cool')
        plot_colored_track('Brake', 'Brake', cmap='Reds')
        plot_colored_track('RPM', 'RPM', cmap='inferno')


# Link dropdowns
year_dropdown.observe(update_races, names='value')
race_dropdown.observe(update_drivers, names='value')
driver_dropdown.observe(update_laps, names='value')
lap_dropdown.observe(show_telemetry, names='value')

update_races(None)

ui = widgets.VBox([year_dropdown, race_dropdown, driver_dropdown, lap_dropdown, output])
display(ui)


('/Users/seanmorris/Library/Caches/fastf1', 122212352)